# Week 7 — A conversation with memory over the course graph

Weeks 3 to 6 built a system that answers one question at a time. This session turns the
Week 6 system into one that holds a **conversation**, and then **remembers** its user in
a later one. Nothing in the model remembers: everything it appears to remember is put
into its context by the code in this notebook.

| Part | What you do |
|---|---|
| 1 | Turn the Week 6 hybrid retriever into a conversation |
| 2 | Rewrite follow-up questions before retrieval |
| 3 | Extract facts about the user, and store them |
| 4 | Use memory in a later conversation |
| 5 | Break it with a stale memory, then repair it |
| 6 | Lab 2, and memory in Lab 3 |

The code is in `week7/conversation.py` and `week7/memory.py`. Read each function when a
part first uses it.

**Backend.** This week runs on **OpenRouter**, since Lightning has been unavailable since
16 October 2026. Create a free account at https://openrouter.ai, create a key under
Settings → API Keys, and put two lines in `.env` at the repository root:

```
LLM_PROVIDER=openrouter
OPENROUTER_API_KEY=your-key-here
```

Then restart the kernel. Free models allow **50 requests a day** per account. One pass
through this notebook makes about 15, and running a cell again makes its requests again.
The embeddings still use your Gemini key.

In [1]:
# Run this notebook from any folder. Week 7's modules live in week7/; they reuse the
# Week 6 retrievers and the modules of Weeks 2 to 5 that those depend on.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3", "week4", "week5", "week6", "week7"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Setup — the course graph from Week 6

This notebook reads the graph that Week 6 built and does **not** rebuild it: rebuilding
empties the database, and your Lab 2 graph with it. If the check below fails, run
`python week6/build_graph.py` from the repository root, then re-run your Lab 2
`ingest.py`.

Neo4j must be running: start Docker Desktop first, then `docker compose up -d` from the
repository root.

In [2]:
from embedding_client import EmbeddingClient
from graph_db import connect
from llm_client import LLMClient
from retrievers import read
from vector_check import require_same_model

driver = connect()
emb = EmbeddingClient()
require_same_model(driver, emb)
print(read(driver, "MATCH (e:Entity) RETURN count(e) AS entities")[0],
      read(driver, "MATCH (c:Chunk) RETURN count(c) AS chunks")[0])

llm = LLMClient()
print(f"answers from {llm.provider} {llm.model}")

{'entities': 372} {'chunks': 183}
answers from openrouter nvidia/nemotron-3-ultra-550b-a55b:free


You should see 372 entities and 183 chunks, and no error from the embedding-model check.

---
## Part 1 — A conversation from a single question

The model keeps **no session**. A conversation is the list of earlier turns, and the
application sends all of it again on every call.

`Conversation` in `conversation.py` keeps the turns. Its `ask` does what Week 6 did for
one question, with one addition: the earlier questions and answers go before the new
prompt, as messages with the roles `user` and `assistant`. The retriever is Week 6's
hybrid, the best of the four on the test set. The answer uses Week 3's grounded
instruction, unchanged.

For now the conversation sends a follow-up to the retriever **as written**. Part 2
changes that.

In [3]:
from conversation import Conversation
from retrievers import hybrid


def retrieve(question):
    return hybrid(driver, emb, question)


talk = Conversation(llm, retrieve, rewrite=False)
turn = talk.ask("What distance does Chroma use by default?", show="full")

════ turn 1 ════════════════════════════════════════════════════════════
question   What distance does Chroma use by default?
context    one request to the model: the instruction, no history, this turn's prompt
──── system │ instruction (170 characters)
You answer questions using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.
──── user │ this turn · sources, task, format, question (4,604 characters)
SOURCES
[1] this one to the taxonomy; the next slide unpacks HNSW, and the one after takes the middle row in detail. Give a decision heuristic rather than a ranking: start from the data platform the project is already committed to. A project already running PostgreSQL should reach for `pgvector` before adding a second system to operate. A project with no database at all and a corpus that fits in memory does not need a server in the first place. State plainly why this course uses Chroma: least op

In [4]:
turn = talk.ask("How do we change it?", show="full")

════ turn 2 ════════════════════════════════════════════════════════════
question   How do we change it?
context    one request to the model: the instruction, 2 history messages, this turn's prompt
──── system │ instruction (170 characters)
You answer questions using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.
──── user │ history · turn 1, the user's message (41 characters)
What distance does Chroma use by default?
──── assistant │ history · turn 1, the model's answer (196 characters)
Chroma's unconfigured default distance metric is squared Euclidean, not cosine similarity. This is why the `hnsw:space` parameter must be set explicitly in the course's `vector_store.py` code. [1]
──── user │ this turn · sources, task, format, question (3,739 characters)
SOURCES
[1] Answer in one sentence and continue. The causes matter here; the label does not. The final row is the one this audience recogn

Each block above is one message, in the order it is sent. All the messages of a turn go to
the model in **one request**: the API receives them as a list. Each header gives two
labels:

- the **role**, which is what the API receives: `system` for the instruction, `user` for
  what the user sent, and `assistant` for the model's own earlier reply
- the **part of the context**: the instruction, the history, or this turn

The second call sends the first question and its answer again, as **history**, before
the new prompt. The model does not remember its first answer: the `assistant` message is
that answer, stored by the application and sent back.

One more turn, printed as a **stack**: one line per message. The history grows by two
messages a turn, a question and an answer, while the instruction stays the same.

In [5]:
turn = talk.ask("Why does the course teach that one?", show="full")

════ turn 3 ════════════════════════════════════════════════════════════
question   Why does the course teach that one?
context    one request to the model: the instruction, 4 history messages, this turn's prompt
──── system │ instruction (170 characters)
You answer questions using only the sources you are given. If the sources do not answer the question, reply exactly: not stated. Never fill a gap from your own knowledge.
──── user │ history · turn 1, the user's message (41 characters)
What distance does Chroma use by default?
──── assistant │ history · turn 1, the model's answer (196 characters)
Chroma's unconfigured default distance metric is squared Euclidean, not cosine similarity. This is why the `hnsw:space` parameter must be set explicitly in the course's `vector_store.py` code. [1]
──── user │ history · turn 2, the user's message (20 characters)
How do we change it?
──── assistant │ history · turn 2, the model's answer (219 characters)
We change it by setting the `hnsw:space` 

**Exercise 1.** Which message carries the sources? Are the earlier turns' sources sent
again with later calls? What would happen to the length of each call if they were?

---
## Part 2 — Rewrite follow-ups before retrieval

Read the answers to turns 2 and 3 above. The retriever received *"How do we change it?"*
as written: a question that names nothing.

The rewrite is one model call. It reads the conversation so far, here turn 1's question
and its answer, then the follow-up, and returns a question that stands on its own. The
cell prints exactly what the call receives, then sends the follow-up to graph by name and
to hybrid, as written and rewritten.

In [6]:
from conversation import REWRITE, rewrite, rewrite_prompt
from retrievers import by_name, named_entities

follow_up = "How do we change it?"
earlier = talk.turns[:1]                          # turn 1 of Part 1's conversation
print("THE REWRITING CALL RECEIVES")
print(f"instruction:\n{REWRITE}\n")
print(f"conversation so far, then the follow-up:\n{rewrite_prompt(earlier, follow_up)}\n")
rewritten = rewrite(llm, earlier, follow_up)
print(f"AND RETURNS\n{rewritten}\n")

for label, question in (("AS WRITTEN", follow_up), ("REWRITTEN", rewritten)):
    facts = by_name(driver, question)
    passages = [s[s.rfind("(chunk "):] for s in hybrid(driver, emb, question) if "(chunk " in s]
    print(f"{label}: {question}")
    print(f"  entities named:  {named_entities(driver, question)}")
    print(f"  graph by name:   {len(facts)} facts")
    for f in facts[:6]:
        print(f"    {f}")
    print(f"  hybrid passages: {', '.join(passages)}\n")

THE REWRITING CALL RECEIVES
instruction:
Rewrite the user's last question so that it can be understood without the conversation. Keep its meaning. Reply with the question only.

conversation so far, then the follow-up:
User: What distance does Chroma use by default?
Assistant: Chroma's unconfigured default distance metric is squared Euclidean, not cosine similarity. This is why the `hnsw:space` parameter must be set explicitly in the course's `vector_store.py` code. [1]
User: How do we change it?

AND RETURNS
How do you change the default distance metric in Chroma?

AS WRITTEN: How do we change it?
  entities named:  []
  graph by name:   0 facts
  hybrid passages: (chunk week-02-lecture.md#25), (chunk week-02-lecture.md#15), (chunk week-01-practical.md#12)

REWRITTEN: How do you change the default distance metric in Chroma?
  entities named:  ['Chroma']
  graph by name:   6 facts
    Embedding uses Chroma. (stated in week-03-practical-plan.md#6)
    Ingest uses Chroma. (stated in week

As written, graph by name is **silent**: the follow-up names no entity. Hybrid searches
for the meaning of *"How do we change it?"* and finds passages from another part of the
course. Rewritten, the question names `Chroma`, and graph by name returns its facts.

Read those facts. None says how to change the distance metric. The rewrite gives the
retriever its **entity**; whether the answer follows depends on what the graph holds.

Below, a conversation that rewrites every follow-up. It prints the rewritten question on
every turn, since a wrong rewrite is otherwise invisible: the retriever, and Week 6's
router if you use one, receive the rewrite, not the follow-up.

In [7]:
talk2 = Conversation(llm, retrieve, rewrite=True)          # rewrite=True is the default
turn = talk2.ask("What distance does Chroma use by default?")
turn = talk2.ask("How do we change it?")

════ turn 1 ════════════════════════════════════════════════════════════
question   What distance does Chroma use by default?
answer     Chroma's unconfigured default distance is squared Euclidean rather than cosine similarity. [1]

════ turn 2 ════════════════════════════════════════════════════════════
question   How do we change it?
rewritten  How do you change the default distance metric in Chroma?
           (what the retriever searched for; the model is still asked the question above)
answer     We change it by explicitly setting the `hnsw:space` parameter in `vector_store.py` to override Chroma's unconfigured default of squared Euclidean distance. [3] This configuration must be set deliberately because the course teaches cosine similarity rather than the default metric. [1]



Compare this answer with turn 2 of Part 1, which searched for the follow-up as written.

**Exercise 2.** Write a follow-up that the rewrite resolves **wrongly**, and ask it in
`talk2`: `talk2.ask("...")`. One way is to refer to something the conversation never
stated. In the lecture, *"Why does the course teach that one?"* became a question about
squared Euclidean distance, because the history never mentioned cosine. How would you
notice a wrong rewrite in a running system?

---
## Part 3 — Extract and store memory

A conversation ends, and its history ends with it. **Memory** is what is kept for a later
conversation: a few facts about the user, not the transcript.

The **application** writes memory. One model call with Week 5's structured output
proposes the facts from a turn; the code stores each with the turn it came from and the
date. The model does not decide to store anything: that would need a tool, and tools are
Week 8.

In [8]:
from memory import EXTRACT, MemoryStore, extract

TURN = ("I missed the Week 4 lecture, so I have not set up vector search in Neo4j yet. "
        "For Lab 1 I used chunks of 300 words. "
        "Which of the Week 6 retrievers should I start with?")
print(EXTRACT, "\n")
facts = extract(llm, TURN)                          # one request
for fact in facts:
    print(f"  {fact}")

From the user's message, list the facts about the user that will still be useful in a later conversation. Leave out the question itself and anything that matters only now. Write each fact as one short sentence about the user. 

  The user missed the Week 4 lecture.
  The user has not set up vector search in Neo4j.
  The user used chunks of 300 words for Lab 1.


Compare the facts with the message: what was kept, and what was left out? A fact
such as *"has not set up vector search yet"* is true now, and not for long. Part 5 returns
to facts like it.

Store the facts. `MemoryStore` in `memory.py` is Week 3's `VectorStore` holding facts
instead of chunks, in `week7/chroma/`, apart from the graph. The date is written as a
course week here, so that Part 5 can move time forward; a real system stores the date.

In [9]:
store = MemoryStore(reset=True)              # an empty store each time this cell runs
part3_ids = store.add(facts, source=TURN, when="Week 6")
for f in store.all():
    print(f"{f['id']}  {f['when']:7}  {f['fact']}")

15777a78  Week 6   The user missed the Week 4 lecture.
ea9f8779  Week 6   The user has not set up vector search in Neo4j.
9abed82b  Week 6   The user used chunks of 300 words for Lab 1.


**Exercise 3.** For your own project, list three facts your system should remember about
its user, and one it must **not** store, with the reason. Every stored fact is sent to the
model again on later calls.

---
## Part 4 — Memory in a later conversation

A later conversation starts with an **empty history**. Without memory, the system knows
nothing about the user: the model has only the course documents.

The store already holds the facts from Part 3. Make them the **profile**: the few facts
placed in every context. Then add two facts kept from an earlier conversation, which are
not in the profile.

In [10]:
store.mark_profile(part3_ids)                           # Part 3's facts become the profile
store.add(["The user runs Neo4j in Docker on a laptop with 8 GB of memory.",
           "The user prefers short answers with a code example."],
          source="an earlier conversation", when="Week 4")
for f in store.all():
    print(f"{f['id']}  {f['when']:7}  {'profile' if f['profile'] else '       '}  {f['fact']}")

15777a78  Week 6   profile  The user missed the Week 4 lecture.
ea9f8779  Week 6   profile  The user has not set up vector search in Neo4j.
9abed82b  Week 6   profile  The user used chunks of 300 words for Lab 1.
8f7f095e  Week 4            The user runs Neo4j in Docker on a laptop with 8 GB of memory.
fbc0143b  Week 4            The user prefers short answers with a code example.


**Without memory, and with it.** The same question opens two new conversations: one
without memory, one with the profile. With memory, the facts about the user are added at
the start of the sources, each written as *"About the user: ..."*, so the grounded answer
can use and cite a fact like any other source. The `memory` line of the log shows which
facts were added.

In [11]:
QUESTION = "Have I set up vector search in Neo4j yet, and what chunk size did I use for Lab 1?"

print("WITHOUT MEMORY")
turn = Conversation(llm, retrieve).ask(QUESTION)

print("WITH MEMORY: the profile")
turn = Conversation(llm, retrieve, memory=lambda q: store.profile()).ask(QUESTION)

WITHOUT MEMORY
════ turn 1 ════════════════════════════════════════════════════════════
question   Have I set up vector search in Neo4j yet, and what chunk size did I use for Lab 1?
answer     Vector search in Neo4j has not been set up yet because Neo4j is introduced in Week 5 [1]. The chunk size used for Lab 1 is not specified in the sources as students must choose and defend their own value [2].

WITH MEMORY: the profile
════ turn 1 ════════════════════════════════════════════════════════════
question   Have I set up vector search in Neo4j yet, and what chunk size did I use for Lab 1?
memory     The user missed the Week 4 lecture.
           The user has not set up vector search in Neo4j.
           The user used chunks of 300 words for Lab 1.
answer     No, you have not set up vector search in Neo4j yet [2]. You used chunks of 300 words for Lab 1 [3].



Without memory, nothing in the sources is about this user. With memory, the same model
answers from the stored facts. Nothing changed in the model: only its context did.

**Explicit or retrieved memory.** The profile is **explicit** memory: always in the
context, whatever the question, so it must stay short. **Retrieved** memory takes, from
the whole store, only the facts nearest in meaning to the question, as RAG does with
documents. Ask a question the profile does not cover, once each way:

In [12]:
QUESTION = "Remind me: how do I run Neo4j, and on what machine?"

print("EXPLICIT MEMORY: the profile")
turn = Conversation(llm, retrieve, memory=lambda q: store.profile()).ask(QUESTION)

print("RETRIEVED MEMORY: the three facts nearest the question")
turn = Conversation(llm, retrieve, memory=lambda q: store.recall(q, k=3)).ask(QUESTION)

EXPLICIT MEMORY: the profile
════ turn 1 ════════════════════════════════════════════════════════════
question   Remind me: how do I run Neo4j, and on what machine?
memory     The user missed the Week 4 lecture.
           The user has not set up vector search in Neo4j.
           The user used chunks of 300 words for Lab 1.
answer     not stated

RETRIEVED MEMORY: the three facts nearest the question
════ turn 1 ════════════════════════════════════════════════════════════
question   Remind me: how do I run Neo4j, and on what machine?
memory     The user runs Neo4j in Docker on a laptop with 8 GB of memory.
           The user has not set up vector search in Neo4j.
           The user used chunks of 300 words for Lab 1.
answer     You run Neo4j in Docker on a laptop with 8 GB of memory. [1]



The profile cannot answer this: the fact is in the store, not in the profile. Retrieval
can hold any number of facts, but finds only those that resemble the question.

So far the store can only add. It keeps growing, and it keeps facts that are no longer
true: Part 5 gives it replace and delete.

**Exercise 4.** Which of your own facts from Exercise 3 belong in the profile, and which
in the store?

---
## Part 5 — A stale memory, and the remedy

Suppose that in Week 6 the store kept *"The user is working on Lab 2."* Lab 2 was due at
the end of Week 7. In Week 8 the user asks what to work on.

This part uses the lecture's prompts exactly: `ASSIST`, a plain instruction without
sources, and `about_user`, which lists what the model knows about the user, with or
without the week each fact was stored.

In [13]:
from memory import ASSIST, about_user

stale_id = store.add(["The user is working on Lab 2."], source=TURN, when="Week 6")[0]
known = [f for f in store.all() if f["profile"] or f["id"] == stale_id]
LATER = "What should I work on this week?"

print(about_user(known, LATER, today="Week 8"), "\n")
for label, today in (("fact only", None), ("with weeks", "Week 8")):
    answer = llm.chat(about_user(known, LATER, today=today), system=ASSIST).text
    print(f"{label}: {answer}\n")

TODAY: Week 8
WHAT YOU KNOW ABOUT THE USER
- (stored in Week 6) The user missed the Week 4 lecture.
- (stored in Week 6) The user has not set up vector search in Neo4j.
- (stored in Week 6) The user used chunks of 300 words for Lab 1.
- (stored in Week 6) The user is working on Lab 2.

QUESTION
What should I work on this week? 

fact only: Catch up on the Week 4 lecture first — it covers vector search setup in Neo4j, which you’ll need for Lab 2. Then configure vector search in your Neo4j instance before continuing Lab 2.

with weeks: Focus on completing Lab 2 this week—make sure your vector search is actually set up in Neo4j since that's likely a blocker. If you missed Week 4, review the lecture on embeddings and vector indexes before finalizing your Lab 2 submission.



In the lecture's runs, with the fact only the answer was *"This week, you should focus on
completing Lab 2 ..."* both times. With the weeks it was once *"check the Week 8 course
schedule"*, once *"finalizing Lab 2 and transitioning to Lab 3"*. The date lets the model
doubt the fact. Only **replacing** the fact removes the error.

Replace it, and ask again:

In [14]:
store.replace(stale_id, "The user submitted Lab 2 at the end of Week 7.",
              source="a Week 7 conversation", when="Week 7")
known = [f for f in store.all() if f["profile"] or f["id"] == stale_id]
print(llm.chat(about_user(known, LATER, today="Week 8"), system=ASSIST).text)

Focus on setting up vector search in Neo4j (which you missed in Week 4) and begin Lab 3 if released, as it likely builds on Lab 2's submissions and requires that infrastructure.


A fact that is wrong, rather than out of date, is deleted. Suppose the user has moved to
a laptop with more memory:

In [15]:
laptop = next(f["id"] for f in store.all() if "8 GB" in f["fact"])
store.delete(laptop)
for f in store.all():
    print(f"{f['id']}  {f['when']:7}  {f['fact']}")
print(f"\nrequests to {llm.provider} from this kernel: {llm.calls}")

15777a78  Week 6   The user missed the Week 4 lecture.
ea9f8779  Week 6   The user has not set up vector search in Neo4j.
9abed82b  Week 6   The user used chunks of 300 words for Lab 1.
fbc0143b  Week 4   The user prefers short answers with a code example.
a084c9a6  Week 7   The user submitted Lab 2 at the end of Week 7.

requests to openrouter from this kernel: 15


**Exercise 5.** `replace` needs to know which fact the new one supersedes. When a later
turn says *"I have submitted Lab 2"*, how would your system find *"The user is working on
Lab 2"* to replace it? One way is to search the store for the nearest fact before adding.
Which of your facts from Exercise 3 will go stale, and when?